In [3]:
import numpy as np
import os
import sys
import joblib

MODE = "test"

try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("ERROR: pip install tensorflow")
    sys.exit(1)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


print("=" * 60)
print("Step 4: Detection & Evaluation")
print("Supervised Autoencoder + BiLSTM + Ensemble")
print("=" * 60)

DATA_DIR = "test_data" if MODE == "test" else "data"
MODEL_DIR = "models"

for fname in ["X_test.npy", "y_test.npy"]:
    if not os.path.exists(os.path.join(DATA_DIR, fname)):
        print(f"ERROR: '{DATA_DIR}/{fname}' not found.")
        sys.exit(1)

X_test = np.load(os.path.join(DATA_DIR, "X_test.npy")).astype(np.float32)
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy")).astype(np.int32)
TIMESTEPS = joblib.load(os.path.join(DATA_DIR, "lstm_timesteps.save")) \
            if os.path.exists(os.path.join(DATA_DIR, "lstm_timesteps.save")) else 10

print(f"Test: {len(X_test):,}  BENIGN: {(y_test==0).sum():,}  ATTACK: {(y_test==1).sum():,}")

def print_results(name, y_true, y_pred, scores=None):
    cm             = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total          = len(y_true)
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"  True  Negatives (BENIGN correct) : {tn:>6,}  ({100*tn/total:.2f}%)")
    print(f"  False Positives (false alarm)     : {fp:>6,}  ({100*fp/total:.2f}%)")
    print(f"  False Negatives (attack missed)   : {fn:>6,}  ({100*fn/total:.2f}%)")
    print(f"  True  Positives (attack caught)   : {tp:>6,}  ({100*tp/total:.2f}%)")
    if scores is not None:
        try:
            print(f"  ROC-AUC : {roc_auc_score(y_true, scores):.4f}")
        except Exception:
            pass
    print()
    print(classification_report(y_true, y_pred, target_names=["BENIGN","ATTACK"]))

# ── 1. Supervised Autoencoder ─────────────────────────────────────────────────
ae_probs = None
ae_path  = None
for ext in [".keras", ".h5"]:
    p = os.path.join(MODEL_DIR, f"autoencoder_model{ext}")
    if os.path.exists(p):
        ae_path = p
        break

if ae_path:
    print(f"\n[1] Loading Supervised Autoencoder: {ae_path}")
    ae      = tf.keras.models.load_model(ae_path, compile=False)
    outputs = ae.predict(X_test, batch_size=1024, verbose=1)

    if isinstance(outputs, dict):
        ae_probs = outputs["classification"].flatten()
    elif isinstance(outputs, (list, tuple)):
        ae_probs = next(o.flatten() for o in outputs if o.shape[-1]==1)
    else:
        ae_probs = outputs.flatten()

    ae_pred = (ae_probs > 0.5).astype(int)
    if MODE == "train":
        print_results("SUPERVISED AUTOENCODER", y_test, ae_pred, ae_probs)
else:
    print("\n[1] Autoencoder not found — run train_autoencoder.py first.")

# ── 2. BiLSTM with Attention ──────────────────────────────────────────────────
lstm_probs = None
y_seq      = None
ae_aligned = None
lstm_path  = None

for ext in [".keras", ".h5"]:
    p = os.path.join(MODEL_DIR, f"bilstm_model{ext}")
    if os.path.exists(p):
        lstm_path = p
        break

if lstm_path:
    print(f"\n[2] Loading BiLSTM: {lstm_path}")
    lstm = tf.keras.models.load_model(
        lstm_path, compile=False)

    n_seq   = len(X_test) - TIMESTEPS
    shape   = (n_seq, TIMESTEPS, X_test.shape[1])
    strides = (X_test.strides[0], X_test.strides[0], X_test.strides[1])
    X_seq   = np.lib.stride_tricks.as_strided(
                  X_test, shape=shape, strides=strides).copy().astype(np.float32)

    # Early detection labels: ATTACK if ANY flow in window is attack
    y_seq = np.array([
        y_test[i + TIMESTEPS - 1]
        for i in range(n_seq)
    ], dtype=np.int32)

    lstm_probs = lstm.predict(X_seq, batch_size=1024, verbose=1).flatten()
    lstm_pred  = (lstm_probs > 0.5).astype(int)
    if MODE == "train":
        print_results("BILSTM WITH ATTENTION (Early Detection)", y_seq, lstm_pred, lstm_probs)

    ae_aligned = ae_probs[TIMESTEPS:] if ae_probs is not None else None
else:
    print("\n[2] BiLSTM not found — run train_bilstm.py first.")

# ── 3. Weighted Ensemble ──────────────────────────────────────────────────────
# AE weight: 0.45  LSTM weight: 0.55
# LSTM gets slightly higher weight because it captures temporal patterns
# which are more discriminative for sequential attack behaviour
if ae_aligned is not None and lstm_probs is not None:
    print("\n[3] Weighted Ensemble: AE(0.45) + BiLSTM(0.55)")
    ens_probs = 0.45 * ae_aligned + 0.55 * lstm_probs
    ens_pred  = (ens_probs > 0.5).astype(int)
    if MODE == "train":
        print_results("WEIGHTED ENSEMBLE (AE + BiLSTM)", y_seq, ens_pred, ens_probs)

def print_summary(preds, name):
    benign = (preds == 0).sum()
    attack = (preds == 1).sum()

    print(f"\n{name} Prediction Summary")
    print("=" * 40)
    print("BENIGN:", benign)
    print("ATTACK:", attack)
    print("TOTAL :", len(preds))


# ── TEST MODE OUTPUTS ─────────────────────────────────────────

import pandas as pd

if MODE == "test":

    def print_summary(preds, name):
        benign = (preds == 0).sum()
        attack = (preds == 1).sum()

        print(f"\n{name} Prediction Summary")
        print("=" * 40)
        print("BENIGN:", benign)
        print("ATTACK:", attack)
        print("TOTAL :", len(preds))

    # Print summaries
    if ae_probs is not None:
        print_summary(ae_pred, "Autoencoder")

    if lstm_probs is not None:
        print_summary(lstm_pred, "BiLSTM")

    if ae_aligned is not None and lstm_probs is not None:
        print_summary(ens_pred, "Ensemble")

    # ── Save predictions with flow data ────────────────────────

    try:
        flow_df = pd.read_csv(os.path.join(DATA_DIR, "processed_flows.csv"))

        # Align for LSTM size difference
        flow_df = flow_df.iloc[-len(ens_pred):].copy()

        flow_df["AE_Pred"]       = ae_pred[-len(flow_df):]
        flow_df["LSTM_Pred"]     = lstm_pred
        flow_df["Ensemble_Pred"] = ens_pred

        flow_df["AE_Prob"]       = ae_probs[-len(flow_df):]
        flow_df["LSTM_Prob"]     = lstm_probs
        flow_df["Ensemble_Prob"] = ens_probs

        # Save full output
        out_path = os.path.join(DATA_DIR, "final_predictions.csv")
        flow_df.to_csv(out_path, index=False)

        print("\nSaved:", out_path)

        # ── Top suspicious flows ───────────────────────────────

        top = flow_df.sort_values(
            by="Ensemble_Prob", ascending=False
        ).head(10)

        top_path = os.path.join(DATA_DIR, "top_suspicious_flows.csv")
        top.to_csv(top_path, index=False)

        print("Saved:", top_path)

    except Exception as e:
        print("Warning: Could not save flow-level predictions:", e)
        
print("detect.py completed successfully!")

TensorFlow version: 2.21.0
Step 4: Detection & Evaluation
Supervised Autoencoder + BiLSTM + Ensemble
Test: 330  BENIGN: 330  ATTACK: 0

[1] Loading Supervised Autoencoder: models\autoencoder_model.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step

[2] Loading BiLSTM: models\bilstm_model.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 677ms/step

[3] Weighted Ensemble: AE(0.45) + BiLSTM(0.55)

Autoencoder Prediction Summary
BENIGN: 324
ATTACK: 6
TOTAL : 330

BiLSTM Prediction Summary
BENIGN: 278
ATTACK: 42
TOTAL : 320

Ensemble Prediction Summary
BENIGN: 294
ATTACK: 26
TOTAL : 320

Saved: test_data\final_predictions.csv
Saved: test_data\top_suspicious_flows.csv
detect.py completed successfully!
